# Imagegeneratie-applicaties bouwen (Azure OpenAI)

Er is meer aan LLM's dan alleen tekstgeneratie. Je kunt ook afbeeldingen genereren op basis van tekstbeschrijvingen. Afbeeldingen als modaliteit zijn nuttig in MedTech, architectuur, toerisme, game-ontwikkeling, marketing en meer. In deze les werken we met de huidige **GPT Image** modellen op Microsoft Foundry.

## Leerdoelen

Aan het einde van deze les kun je:

- Uitleggen wat imagegeneratie is en waar het nuttig voor is.
- De `gpt-image` modelreeks begrijpen en weten hoe deze verschilt van de legacy DALL·E-modellen.
- Een imagegeneratie-applicatie bouwen en afbeeldingen bewerken.

## Wat is imagegeneratie?

Imagegeneratiemodellen creëren afbeeldingen uit een tekstprompt. Moderne modellen zoals `gpt-image` leren tijdens training de relatie tussen tekst en afbeeldingen, en zetten dan iteratief willekeurige ruis om in een afbeelding die bij je prompt past.

Twee bekende modelreeksen voor afbeeldingen zijn:

- **`gpt-image` (OpenAI)** — de huidige generatie die in deze les wordt gebruikt. Het ondersteunt tekst-naar-beeld generatie en afbeeldingsbewerking (inpainting met een masker).
- **Midjourney** — een populair model van derden met een eigen service en een workflow via Discord.

> De oudere OpenAI afbeeldingsmodellen — **DALL·E 2** en **DALL·E 3** — zijn legacy. DALL·E 3 is niet langer beschikbaar voor nieuwe implementaties, en functies zoals `create_variation` bestonden alleen in DALL·E 2. Gebruik voor nieuwe toepassingen de `gpt-image` modellen.

Op Microsoft Foundry is **`gpt-image-2`** het nieuwste en meest capabele afbeeldingsmodel en wordt het aanbevolen als standaard. `gpt-image-1.5` en `gpt-image-1-mini` zijn ook algemeen beschikbaar.

> **Belangrijk:** `gpt-image` modellen retourneren de gegenereerde afbeelding als **base64** (`b64_json`), niet als URL. Je code decodeert de base64-string naar bytes en slaat deze op — er is geen afbeeldings-URL om te downloaden.


## Je eerste applicatie voor beeldgeneratie bouwen

Dus wat heb je nodig om een applicatie voor beeldgeneratie te bouwen? Je hebt de volgende bibliotheken nodig:

- **python-dotenv**, het wordt sterk aangeraden om deze bibliotheek te gebruiken om je geheimen in een *.env* bestand te bewaren, gescheiden van de code.
- **openai**, deze bibliotheek gebruik je om te communiceren met de OpenAI API.
- **pillow**, om met afbeeldingen te werken in Python.

Als je dat nog niet gedaan hebt, volg de instructies op de [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) pagina om een Microsoft Foundry resource en model aan te maken. Selecteer **gpt-image-2** als model (het nieuwste Azure OpenAI beeldmodel; DALL·E is legacy).

1. Maak een bestand *.env* aan met de volgende inhoud:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Vind deze informatie in het Microsoft Foundry portaal voor jouw resource onder het gedeelte "Deployments".



1. Verzamel de bovenstaande bibliotheken in een bestand genaamd *requirements.txt* zoals volgt:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Maak vervolgens een virtuele omgeving aan en installeer de bibliotheken:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Gebruik voor Windows de volgende opdrachten om je virtuele omgeving te maken en te activeren:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Voeg de volgende code toe in een bestand genaamd *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # importeer dotenv
    dotenv.load_dotenv()

    # configureer Azure OpenAI (Microsoft Foundry) client.
    # Afbeeldingsmodellen hebben een recente API-versie nodig — controleer de Foundry-documentatie voor de versie die jouw model vereist.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Maak een afbeelding met de beeldgeneratie-API
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Voer hier je prompttekst in
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # bijv. "gpt-image-2"
        )
        # Stel de map in voor de opgeslagen afbeelding
        image_dir = os.path.join(os.curdir, 'images')

        # Als de map niet bestaat, maak deze aan
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialiseer het afbeeldingspad (let op, het bestandstype moet png zijn)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modellen retourneren de afbeelding als base64 (b64_json), niet als URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Toon de afbeelding in de standaard afbeeldingsviewer
        image = Image.open(image_path)
        image.show()

    # vang uitzonderingen op
    except BadRequestError as err:
        print(err)

    ```

Laten we deze code uitleggen:

- Eerst importeren we de benodigde bibliotheken, waaronder de OpenAI bibliotheek, de dotenv bibliotheek, de base64 module, en de Pillow bibliotheek.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Vervolgens laden we de omgevingsvariabelen uit het *.env* bestand.

    ```python
    # importeer dotenv
    dotenv.load_dotenv()
    ```

- Daarna configureren we de Azure OpenAI (Microsoft Foundry) client.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Vervolgens genereren we de afbeelding:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Voer hier uw prompttekst in
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` modellen geven de afbeelding als een **base64** string terug in `data[0].b64_json`. We decoderen deze naar bytes en schrijven het naar een bestand — er is geen URL om te downloaden.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Tot slot openen we de afbeelding en gebruiken de standaard afbeeldingsviewer om deze te tonen:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Meer details over het genereren van de afbeelding

Laten we de parameters van `images.generate` bekijken:

- **prompt**, is de tekstprompt die gebruikt wordt om de afbeelding te genereren. Hier is het "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, is de grootte van de gegenereerde afbeelding (bijvoorbeeld `1024x1024`, `1536x1024`, `1024x1536`, of `"auto"`).
- **n**, is het aantal gegenereerde afbeeldingen. Hier genereren we er één.
- **model**, is de naam van je beeldmodel-deployment (bijvoorbeeld `gpt-image-2`).

> Beeldmodellen gebruiken geen `temperature` parameter — dat is voor tekstgeneratie. Om variatie te krijgen, roep je de API opnieuw aan; om variatie te verminderen, maak je je prompt specifieker.

## Extra mogelijkheden van beeldgeneratie

Je hebt gezien hoe je een afbeelding genereert met een paar regels Python. `gpt-image` modellen kunnen ook een **bestaande afbeelding bewerken**. Door een afbeelding, een optioneel **masker** (dat het te wijzigen gebied markeert), en een prompt te geven, kun je een deel van een afbeelding aanpassen — bijvoorbeeld een hoed toevoegen aan onze konijn.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# bewerkingen worden ook teruggegeven als base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

De basisafbeelding bevat alleen het konijn; de uiteindelijke afbeelding heeft een hoed op het konijn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
